# 🏠 House Price Prediction
**Model:** Ridge + Lasso + ElasticNet stacked ensemble  
**Dataset:** Ames Housing (Kaggle House Prices)  
**Target:** SalePrice (log-transformed)  

**Techniques:**
- Log-transform on target to normalise skew
- Out-of-fold stacking: base learners trained on CV folds, meta-learner on OOF predictions
- SHAP for feature attribution

In [ ]:
!git clone https://github.com/Aurovindhya/ml-portfolio-suite.git
%cd ml-portfolio-suite
!pip install -q scikit-learn pandas numpy matplotlib seaborn shap

In [ ]:
# Download Ames Housing from OpenML or generate sample
import pandas as pd
import numpy as np
import os

try:
    from sklearn.datasets import fetch_openml
    housing = fetch_openml(name='house_prices', as_frame=True)
    df = housing.frame
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    if 'saleprice' not in df.columns:
        df['saleprice'] = housing.target.astype(float)
    print(f'OpenML dataset loaded: {df.shape}')
except Exception:
    # Synthetic fallback
    np.random.seed(42)
    n = 1460
    df = pd.DataFrame({
        'overall_qual':    np.random.randint(1, 11, n),
        'gr_liv_area':     np.random.randint(500, 4000, n).astype(float),
        'garage_cars':     np.random.randint(0, 5, n).astype(float),
        'garage_area':     np.random.randint(0, 1400, n).astype(float),
        'total_bsmt_sf':   np.random.randint(0, 3000, n).astype(float),
        'first_flr_sf':    np.random.randint(300, 3000, n).astype(float),
        'full_bath':       np.random.randint(0, 4, n).astype(float),
        'tot_rms_abv_grd': np.random.randint(2, 14, n).astype(float),
        'year_built':      np.random.randint(1900, 2010, n).astype(float),
        'year_remod_add':  np.random.randint(1950, 2010, n).astype(float),
        'lot_area':        np.random.randint(1500, 50000, n).astype(float),
        'mas_vnr_area':    np.random.choice([0]*800 + list(range(1, 1400)), n).astype(float),
    })
    df['saleprice'] = (
        df['overall_qual'] * 20000
        + df['gr_liv_area'] * 80
        + np.random.normal(0, 20000, n)
    ).clip(50000, 750000)
    print(f'Synthetic dataset: {df.shape}')

os.makedirs('data', exist_ok=True)
df.to_csv('data/house_prices.csv', index=False)
df.describe()

In [ ]:
# Price distribution (log-transform)
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(df['saleprice'], bins=50); ax1.set_title('SalePrice (raw)')
ax2.hist(np.log1p(df['saleprice']), bins=50); ax2.set_title('log1p(SalePrice)')
plt.tight_layout(); plt.show()
print('Log-transform normalises the skew — critical for linear models')

In [ ]:
import sys; sys.path.insert(0, '.')
from modules.regression.house_prices.model import train
metrics = train('data/house_prices.csv')
print('Stacked ensemble metrics:', metrics)

In [ ]:
# Visualise ensemble weights
import pickle
import numpy as np

with open('weights/house_price.pkl', 'rb') as f:
    bundle = pickle.load(f)

meta_coefs = bundle['meta'].coef_
labels = ['Ridge', 'Lasso', 'ElasticNet']
plt.bar(labels, meta_coefs, color=['#2196F3','#FF5722','#4CAF50'])
plt.title('Meta-learner weights (how much each base model contributes)')
plt.ylabel('Coefficient'); plt.show()

In [ ]:
from modules.regression.house_prices.model import predict

sample = {
    'overall_qual': 8, 'gr_liv_area': 2000, 'garage_cars': 2,
    'garage_area': 600, 'total_bsmt_sf': 1200, 'first_flr_sf': 1200,
    'full_bath': 2, 'tot_rms_abv_grd': 8, 'year_built': 2005,
    'year_remod_add': 2010, 'lot_area': 9000, 'mas_vnr_area': 150
}
price, ms = predict(sample)
print(f'Predicted price: ${price:,.0f} ({ms:.1f}ms)')